<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 1: </b>Primera neurona</h2>
<br>
En este notebook entrenamos nuestra primera neurona para que haga una labor de predicción bastante particular...

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import utils

# Fijamos una seed para que sea reproducible los resultados, usando la respuesta de la vida (revisar Hitchhiker's Guide to the Galaxy)
torch.manual_seed(42)

----

<br>

## **Parte 0:** Nuestro objetivo

<div style="float: right; width: 36%; min-width: 170px; max-width: 300px; margin: 4px 0px 12px 30px;">
  <img src="assets/mensajes.png" width="100%" alt="Celular con mensajes sin responder"
       style="display: block; transform: rotate(-1.5deg);
              filter: drop-shadow(0px 12px 20px rgba(41, 196, 217, 0.35));"/>
</div>

Julián y Manuela llevan una relación feliz. Sin embargo, Manuela tiende a ser una persona muy atenta y demandante en la comunicación (intensa), por lo que a Julián le interesa anticipar cuántos mensajes de WhatsApp se le van a acumular mientras no contesta, con el objetivo de responder antes de que se arme un problema (tóxica).

El objetivo de este notebook es construir un modelo de redes neuronales capaz de predecir **cuántos mensajes habrá mandado Manuela** a partir de **los minutos que Julián lleva sin responder**, usando el historial de la conversación.

<div style="clear: both;"></div>

----

<br>

## **Parte 1:** Los datos

Empezaremos subiendo los datos sacados del historial de chat donde:

- **Entrada** ($x$): minutos que Julián lleva callado.
- **Salida** ($y$): mensajes que Manuela ha enviado hasta ese momento.

Queremos un modelo de la forma:

$$
F:x \rightarrow y
$$

que prediga cuántos mensajes enviará Manuela según el tiempo que Julián lleve sin responder.

In [ ]:
# Entrada (x): minutos que Julián lleva sin responder
minutos = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0], [7.0], [8.0]])

# Salida esperada (y): mensajes que Manuela ya mandó
mensajes = torch.tensor([[1.0], [2.0], [3.0], [4.0], [6.0], [11.0], [23.0], [50.0]])

utils.dibujar_datos(minutos, mensajes, titulo="Historial cantidad de mensajes")

Podemos notar un crecimiento preocupante de los mensajes a medida de que va pasando el tiempo.

----

<br>

## **Parte 2:** La neurona

Iniciaremos con una neurona simple, cuya ecuación tendrá la forma:

$$
y = \text{ReLU}(w \cdot x + b)
$$

Veremos dos formas de implementarla, una manera sencilla y otra usando clases personalizadas.


La primera forma es bastante directa, es colocar de forma secuencial los cálculos que se harán en la neurona

In [ ]:
neurona_sequential = nn.Sequential(
    nn.Linear(1, 1), # w*x +b
    nn.ReLU()
)

Por otro lado usando herencia se puede realizar la misma red neuronal pero permitiendo en un futuro mucho más personalización

In [ ]:
class Neurona(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(1, 1) # w*x + b
        self.relu = nn.ReLU() 

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        return x

A partir de ahora usaremos esta segunda forma en todo el notebook

Ahora vamos a inicializar el modelo, definir su función de pérdida y su optimizador.

In [ ]:
modelo = Neurona()
funcion_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

Ahora sí, comenzaremos con el proceso de entrenamiento, el cual sigue el siguiente flujo:

**Borrar gradientes → Predecir → Medir error → Calcular gradientes → Actualizar parámetros → Repetir**

Este ciclo se repite durante varias épocas (*epochs*) que fueron definidas.

In [ ]:
historial = []

for epoca in range(2000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(minutos)                     # 2. predecir
    perdida = funcion_perdida(predicciones, mensajes)  # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial.append(perdida.item())
    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1:4d} — pérdida: {perdida.item():.4f}")

# Graficando la perdida
utils.dibujar_perdida(historial)

Ya tenemos el modelo entrenado, ahora veamos cómo quedó la función

In [ ]:
utils.dibujar_ajuste(minutos, mensajes, modelo, etiqueta="Neurona",
                     titulo="Predicción de una sola neurona")

Julián al ver este modelo nos miró raro, por lo tanto tocará hacerle unos ajustes (poner más neuronas).

Representación de la mirada de Julián

![julian](assets/julian.png)

----

<br>

## **Parte 3:** La red neuronal

Ya tuvimos un primer intento de modelo usando una neuronal, que dió como resultado una recta con un doblez, pero eso no es suficiente.

Ahora usamos varias neuronas en una capa oculta, más una capa que combina sus salidas. Cada neurona aporta un doblez y juntas lograrán aproximar una mejor curva.

In [ ]:
class RedNeuronal(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(1, 5)
        self.relu = nn.ReLU() 
        self.layer_2 = nn.Linear(5, 1)
    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

In [ ]:
modelo = RedNeuronal()
funcion_perdida = nn.MSELoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

historial = []

for epoca in range(2000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(minutos)                     # 2. predecir
    perdida = funcion_perdida(predicciones, mensajes)  # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial.append(perdida.item())
    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1:4d} — pérdida: {perdida.item():.4f}")

# Graficando la perdida
utils.dibujar_perdida(historial)

El modelo se ve con una buena curva de aprendizaje, ahora veamos que tal aproxima los datos

In [ ]:
utils.dibujar_ajuste(minutos, mensajes, modelo, etiqueta="Red neuronal",
                     titulo="Predicción de la red neuronal")

Con este modelo, Julián ya puede aproximar cuántos mensajes le mandará Manuela según el tiempo que lleve sin responder. Así podrá anticipar el momento crítico, justo antes del desastre, y responder a tiempo para evitar una pelea.

En el próximo notebook veremos cómo usar la red neuronal con unos datos más comeplejos: las imagenes.

-----

<br>

### **Siguiente notebook:**

[`02_clasificador_imagenes.ipynb`](02_clasificador_imagenes.ipynb)

### <font color="#29c4d9">**Notebook 1 listo.**</font>

<br>

---

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>